In [1]:
# =============================================================================
# 1: DRIVE MOUNT
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =============================================================================
# # 4: MULTIMODAL DATA FUSION & INFERENCE ENGINE
# =============================================================================
#
# PURPOSE:
# This cell functions as the project's primary "Prediction Engine." It bridges
# Google Earth Engine's (GEE) cloud data with the local TensorFlow environment
# to generate live probability heatmaps of mining activity.
#
# THE 7-BAND STACK (Fusion Logic):
#   - Sentinel-2 Optical: Blue, Green, Red, and NIR bands for spectral clarity.
#   - NDVI: Added to improve differentiation between mining pits and natural gaps.
#   - Sentinel-1 SAR: VV and VH backscatter to provide texture and cloud-resiliency.
#
# KEY OPERATIONS:
#   - Normalization: Scales raw 16-bit satellite values to 0.0-1.0 float ranges.
#   - Tensor Reshaping: Formats the GEE output into the 256x256x7 input required.
#   - High-Precision Logic: Applies the v6 optimized 0.25 probability threshold.
#
# This directly supports:
#   - Hypothesis H1: Fusing Radar and Optical data improves detection accuracy.
#
# Author: ATD | Johns Hopkins University | April 2026
# ==============================================================================

def run_live_inference(aoi, threshold=0.25):
    """
    Performs multimodal inference by fusing Sentinel-1 and Sentinel-2 data.
    Directly supports Hypothesis H1: Accuracy of fused SAR/Optical detection.
    """
    # 1. Fetch Sentinel-2 + NDVI
    s2_col = (ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
              .filterBounds(aoi)
              .filterDate('2025-01-01', '2026-04-26')
              .sort('CLOUDY_PIXEL_PERCENTAGE'))

    s2_img = s2_col.first()
    s2_bands = s2_img.select(['B2', 'B3', 'B4', 'B8']).float()
    ndvi = s2_bands.normalizedDifference(['B8', 'B4']).rename('NDVI').float()

    # 2. Fetch Sentinel-1 SAR (VV/VH)
    s1_col = (ee.ImageCollection('COPERNICUS/S1_GRD')
              .filterBounds(aoi)
              .filterDate('2025-01-01', '2026-04-26'))

    # Handle radar gaps with a median composite
    s1_bands = s1_col.median().select(['VV', 'VH']).float().unmask(-15)

    # 3. Assemble 7-Band Stack (Mirroring Training Library)
    final_stack = s2_bands.addBands(ndvi).addBands(s1_bands)

    # 4. Export to Local Array for Model
    # We use a 256x256 patch size to match the U-Net input
    patch = geemap.ee_to_numpy(final_stack, region=aoi, scale=5)

    # 5. Preprocessing & Prediction
    # Normalize (dividing by 2000 as per GOLD Training Pipeline)
    input_tensor = np.expand_dims(patch / 2000.0, axis=0)
    raw_prediction = best_model.predict(input_tensor, verbose=0)[0, :, :, 0]

    # 6. Post-Processing
    # Apply Threshold + Morphological Closing to reduce speckle noise
    binary_map = (raw_prediction > threshold).astype(np.uint8)
    refined_map = ndimage.binary_closing(binary_map, structure=np.ones((5,5)))

    cloud_pct = s2_img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
    print(f"✅ Inference Successful (Cloud Cover: {cloud_pct:.2f}%)")

    return refined_map

In [3]:
# =============================================================================
# # 5: INDIGENOUS TERRITORY (TI) SPATIAL FILTERING
# =============================================================================
#
# PURPOSE:
# This cell defines the geographical parameters of the study. It automates
# the selection of "Protected Area" geometries to ensure the search for
# illegal mining is legally and contextually bounded.
#
# KEY OPERATIONS:
#   - Asset Loading: Fetches the 'Tis_TerritoriosIndigenas' GEE asset.
#   - Geometry Intersection: Uses .filterBounds() to strictly isolate known
#     mining activity that occurs INSIDE protected red boundaries.
#   - Point Sampling: Selects distinct internal mine coordinates to serve
#     as "blind" test-beds for the AI validation.
#
# This directly supports:
#   - Capstone Goal: Monitor encroachment in legally protected Brazilian territories.
#
# Author: ATD | Johns Hopkins University | April 2026
# =============================================================================
import ee
import geemap
from google.colab import auth

# 1. Authenticate your Google Account
auth.authenticate_user()

# 2. Initialize with your specific Project ID
try:
    ee.Initialize(project='trusty-hub-437505-e5')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='trusty-hub-437505-e5')

# 3. Load the TI Asset
ti_asset_path = 'projects/trusty-hub-437505-e5/assets/Tis_TerritoriosIndigenas'
indigenous_territories = ee.FeatureCollection(ti_asset_path)

# 4. Create and display the map
Map = geemap.Map()
Map.centerObject(indigenous_territories, 5)
Map.addLayer(indigenous_territories, {'color': 'red'}, 'Indigenous Territories (TIs)')
Map

Map(center=[1.895907564531574, -63.271790283570446], controls=(WidgetControl(options=['position', 'transparent…

In [4]:
import numpy as np
import tensorflow as tf
from scipy import ndimage

# 1. Load your Final Optimized Model
model_path = '/content/drive/MyDrive/mining_unet_final.h5'
best_model = tf.keras.models.load_model(model_path, compile=False)

def run_territory_scan(geometry, threshold=0.25):
    """
    Scans the provided TI geometry for mining footprints
    using the v6 high-precision logic.
    """
    print(f"🚀 Initializing Scan for: {ti_asset_path}...")

    # This function will fetch the Sentinel-1/2 stack for the area
    # Note: For large TIs, we process the centroid or a sampled grid
    # to avoid GEE memory limits.

    # Placeholder for the extraction & prediction loop:
    # 1. Fetch fused S1/S2 data from GEE
    # 2. Normalize (images / 2000)
    # 3. model.predict()
    # 4. Apply 0.25 Threshold + ndimage.binary_closing

    print("✅ Model loaded and ready for tiled inference.")
    print(f"🎯 Detection Strategy: High-Precision (Threshold: {threshold})")

# Run a test scan on the loaded TIs
run_territory_scan(indigenous_territories.geometry())

🚀 Initializing Scan for: projects/trusty-hub-437505-e5/assets/Tis_TerritoriosIndigenas...
✅ Model loaded and ready for tiled inference.
🎯 Detection Strategy: High-Precision (Threshold: 0.25)


In [31]:
# =============================================================================
# # 6: MORPHOLOGICAL POST-PROCESSING & MAP VISUALIZATION
# =============================================================================
#
# PURPOSE:
# This cell transforms raw, "speckled" AI predictions into clean, actionable
# geographic features and presents them in an interactive dashboard.
#
# KEY OPERATIONS:
#   - Scipy Binary Closing: Uses ndimage.binary_closing to fill small holes
#     in predictions, creating cohesive mining footprints.
#   - Multilayer Visualization: Overlays the AI (Magenta), Ground Truth (Yellow),
#     and TI Boundaries (Red) onto high-resolution Sentinel-2 imagery.
#   - Target Marker Validation: Places a persistent "X" marker at the centroid
#     of the detection to facilitate rapid visual audit by human analysts.
#
# This directly supports:
#   - Capstone Utility: Demonstrates a workflow that translates complex
#     probability data into high-confidence visual intelligence.
#
# Author: ATD | Johns Hopkins University | April 2026
# =============================================================================

def find_two_internal_mines_with_toggles():
    # 1. Load Assets
    ground_truth_asset = 'projects/trusty-hub-437505-e5/assets/mining_cumulative_2018_2024'
    known_mines = ee.FeatureCollection(ground_truth_asset)

    # 2. FILTER: Only mines that intersect the TI boundaries
    internal_mines = known_mines.filterBounds(indigenous_territories.geometry())

    # Select 2 guaranteed internal targets
    targets = internal_mines.randomColumn().sort('random').limit(2).getInfo()['features']

    Map_Internal = geemap.Map()

    for i, target in enumerate(targets):
        geom = ee.Geometry(target['geometry']).centroid(1)
        aoi = geom.buffer(800).bounds()
        coords = geom.coordinates().getInfo()

        # 3. Fetch Clear Imagery (2024-2026)
        s2_img = (ee.ImageCollection("COPERNICUS/S2_HARMONIZED")
                  .filterBounds(aoi)
                  .filterDate('2024-01-01', '2026-04-26')
                  .sort('CLOUDY_PIXEL_PERCENTAGE')
                  .first())

        # 4. Run "Blind" v6 Inference (0.25 Threshold)
        result = run_live_inference(aoi)
        img_proj = s2_img.select('B2').projection()
        mining_ee = geemap.numpy_to_ee(result, crs=img_proj.wkt(), transform=img_proj.getInfo()['transform'])

        # 5. Build Map Layers
        Map_Internal.addLayer(s2_img, {'bands':['B4','B3','B2'], 'max':2500}, f'Sat Img {i+1}')

        # Ground Truth Polygons (Yellow)
        Map_Internal.addLayer(known_mines.filterBounds(aoi), {'color': 'yellow', 'weight': 2}, f'Ground Truth {i+1}')

        # AI Predicted Footprint (Magenta)
        Map_Internal.addLayer(mining_ee.mask(mining_ee), {'palette':['#FF00FF']}, f'v6 AI Detection {i+1}')

        # Reference Marker
        Map_Internal.add_marker(location=[coords[1], coords[0]], tooltip=f"Internal Mine {i+1}")

    # 6. Add the TI Boundary (Red) and ensure it's SELECTABLE
    Map_Internal.addLayer(indigenous_territories, {'color': 'red', 'width': 2}, 'TI Boundary', opacity=0.4)

    # 7. Final UI Config
    Map_Internal.centerObject(geom, 15)
    Map_Internal.add_layer_control()

    print("✅ Strict Internal Audit Complete. All layers are now toggleable.")
    display(Map_Internal)

find_two_internal_mines_with_toggles()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
✅ Inference Successful (Cloud Cover: 0.00%)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
✅ Inference Successful (Cloud Cover: 4.76%)
✅ Strict Internal Audit Complete. All layers are now toggleable.


Map(center=[3.3006464784354623, -63.67402297516231], controls=(WidgetControl(options=['position', 'transparent…